In [1]:
# 20260724:
# Step1, attach the tcr calling from trust4:

In [1]:
#!/usr/bin/env python
"""
Per-sample clonotype definition, then combine.

For each subdir: read TRUST4 AIRR -> rename barcodes -> index_chains/chain_qc ->
keep cells with a primary cdr3_aa -> ir_dist(aa, identity) ->
define_clonotype_clusters -> prefix clone_id with the sample name.
Only then are all samples concatenated into one object.

Consequence: clonotypes are patient-private by construction. No cross-sample
transitive chaining, no public-clone merging. Cross-sample sharing is still
recoverable by grouping on the 'primary_cdr3_aa' column.

Input layout:
    trust4_inputdir/<subdir>/XXX_<subdir>_CD45_CD8_TRUST4_barcode_airr.tsv

Barcode renaming: cell_id -> f"{cell_id}-{subdir}_CD45"

Requires: scirpy >= 0.13, anndata >= 0.9, pandas, numpy.
"""

import os
import glob

import numpy as np
import pandas as pd
import anndata as ad
import scirpy as ir




#!/usr/bin/env python
"""
Per-sample clonotype definition, then combine.

For each subdir: read TRUST4 AIRR -> rename barcodes -> index_chains/chain_qc ->
keep cells with a primary cdr3_aa -> ir_dist(aa, identity) ->
define_clonotype_clusters -> prefix clone_id with the sample name.
Only then are all samples concatenated into one object.

Consequence: clonotypes are patient-private by construction. No cross-sample
transitive chaining, no public-clone merging. Cross-sample sharing is still
recoverable by grouping on the 'primary_cdr3_aa' column.

Input layout:
    trust4_inputdir/<subdir>/XXX_<subdir>_CD45_CD8_TRUST4_barcode_airr.tsv

Barcode renaming: cell_id -> f"{cell_id}-{subdir}_CD45"

Requires: scirpy >= 0.13, anndata >= 0.9, pandas, numpy.
"""

import os
import glob

import numpy as np
import pandas as pd
import anndata as ad
import scirpy as ir




# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
INPUT_DIR = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/TRUST4_output"
# subdir already ends in _CD45 (e.g. "patient10_T02_CD45"), so the glob must not
# add another one. This pattern matches XXX_<subdir>_CD8_TRUST4_barcode_airr.tsv
# as well as older XXX_<subdir>_CD45_CD8_... layouts.
FILE_GLOB = "*{subdir}*_TRUST4_barcode_airr.tsv"
BARCODE_SUFFIX_TMPL = "-{subdir}"

# Study label applied to every sample, and the prefix used to build sample names.
STUDY = "chiffelle"

# Arm whose primary CDR3 defines the clonotype.
#   "VDJ_1" -> primary beta chain (matches the training-data definition)
#   None    -> keep any cell with a primary CDR3 in EITHER arm (beta preferred)
PRIMARY_ARM = "VDJ_1"

# Clonotype-graph arms, matching the true-TCR calls exactly:
CLONE_ID_ARMS = "any"   # define_clonotypes  -> clone_id       (auxiliary)
CC_ARMS = "all"         # define_clonotype_clusters -> cc_aa_identity  (FINAL basis)
DUAL_IR = "primary_only"


# AIRR amino-acid junction field name as it appears in the TRUST4 files.
# Standard AIRR uses 'junction_aa' (includes flanking Cys/Phe); some pipelines
# expose 'cdr3_aa' (trimmed). Must match whatever the clonotypes were defined on
# in the true-TCR training data.
SEQ_FIELD = "junction_aa"


# Write each per-sample object (with its ir_dist matrices intact) to disk.
# Those matrices do NOT survive concat, so save here if you want to run
# clonotype_network / clonal_expansion per sample later.
SAVE_PER_SAMPLE = False
PER_SAMPLE_DIR = "per_sample_h5ad"

OUTPUT_DIR = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle"
OUT_PREFIX = "trust4_chiffelle"

# Reference table used ONLY to derive the correct per-sample barcode suffix.
# Must contain columns 'cell_barcode' and 'sample'. The suffix appended in
# load_sample is taken from here (part after the first '-'), so the generated
# cell_id matches cell_barcode exactly. Set to None to fall back to "-{subdir}".
ENSEMBLE_mode    = "rankMin_study"
REFERENCE_DF = f"/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/ICB_chiffelle_5case_inference_6AllasTrain/unified_cell_predictions/raw_cell_scores/scTRP_simclr_{ENSEMBLE_mode}_chiffelle.csv"

# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def make_sample_name(subdir, study=STUDY):
    """'patient10_T02_CD45' -> 'chiffelle_Patient10_T02' (first char upper only)."""
    base = subdir.removesuffix("_CD45")
    return f"{study}_{base[:1].upper()}{base[1:]}"
 
 
def sample_to_subdir(sample, study=STUDY):
    """Inverse: 'chiffelle_Patient10_T01' -> 'patient10_T01'."""
    base = sample.removeprefix(f"{study}_")
    return base[:1].lower() + base[1:]
 
 
def build_suffix_map(reference_csv):
    """From the reference table, return {subdir: (sample, suffix)}.
    suffix = cell_barcode after the FIRST '-'; asserts one suffix per sample."""
    ref = pd.read_csv(reference_csv, usecols=["cell_barcode", "sample"])
    ref["_suffix"] = ref["cell_barcode"].astype(str).str.split("-", n=1).str[1]
 
    resolved = {}
    for sample, grp in ref.groupby("sample"):
        suffixes = grp["_suffix"].dropna().unique()
        if len(suffixes) != 1:
            raise ValueError(
                f"Sample {sample!r} has {len(suffixes)} distinct barcode suffixes "
                f"{list(suffixes)[:5]}; expected exactly 1."
            )
        resolved[sample_to_subdir(sample)] = (sample, suffixes[0])
 
    print(f"Derived suffixes for {len(resolved)} samples from {reference_csv}")
    return resolved
 
 
def find_subdirs(input_dir):
    subs = sorted(d for d in os.listdir(input_dir)
                  if os.path.isdir(os.path.join(input_dir, d)))
    if not subs:
        raise FileNotFoundError(f"No subdirectories found under {input_dir!r}.")
    return subs
 

def load_sample(input_dir, subdir, sample, suffix):
    """Read one AIRR tsv, append the df-derived suffix, tag sample/study."""
    pattern = os.path.join(input_dir, subdir, FILE_GLOB.format(subdir=subdir))
    matches = glob.glob(pattern)
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 file matching {pattern!r}, found {len(matches)}: {matches}"
        )
    path = matches[0]
 
    adata = ir.io.read_airr(path)   # obs indexed by the file's cell_id column
 
    file_ids = set(pd.read_csv(path, sep="\t", usecols=["cell_id"])["cell_id"].astype(str))
    if set(adata.obs_names.astype(str)) != file_ids:
        print(f"  [warn] {subdir}: obs_names differ from file cell_id")
 
    if suffix is None:                       # fallback: "-{subdir}"
        suffix = BARCODE_SUFFIX_TMPL.format(subdir=subdir).lstrip("-")
        sample = make_sample_name(subdir)
 
    adata.obs_names = adata.obs_names.astype(str) + "-" + suffix   # == cell_barcode
    adata.obs["cell_id"] = adata.obs_names
    adata.obs["sample"] = sample                      # e.g. chiffelle_Patient10_T01
    adata.obs["study"] = STUDY
    print(f"[{subdir}] {os.path.basename(path)}: {adata.n_obs} cells -> {sample}")
    return adata


def get_primary_cdr3(adata, arm):
    """Series (index = obs_names) of the primary SEQ_FIELD (junction_aa) for the arm."""
    df = ir.get.airr(adata, [SEQ_FIELD], arm)
    cols = [c for c in df.columns if SEQ_FIELD in c]
    if len(cols) != 1:
        raise RuntimeError(f"Expected one {SEQ_FIELD} column, got: {list(df.columns)}")
    return df[cols[0]]
 
 
def junction_to_cdr3(seq):
    """Trim an AIRR junction_aa to cdr3_aa by removing the flanking conserved
    anchors (leading Cys, trailing Phe/Trp): cdr3 = junction[1:-1].
 
    Only trims strings that look like a real junction (start 'C', end 'F'/'W',
    len >= 3); anything else (NaN, malformed, already-trimmed) is returned as-is.
    """
    if not isinstance(seq, str) or len(seq) < 3:
        return seq
    if seq[0] == "C" and seq[-1] in ("F", "W"):
        return seq[1:-1]
    return seq                                    # doesn't match the anchor pattern
 
 
def write_tcr_seq_cols(adata):
    """Write per-chain V/J gene + junction into obs (VJ_1_*, VDJ_1_*, ...), a
    convenience `cdr3_aa` = primary beta junction (VDJ_1_<SEQ_FIELD>), and a
    trimmed `cdr3_aa_trimmed` (junction with flanking Cys/Phe|Trp removed).
    Returns the list of newly added obs column names."""
    seq_df = ir.get.airr(adata, ["v_call", SEQ_FIELD, "j_call"])   # multi-chain form
    seq_df = seq_df.reindex(adata.obs_names)
    adata.obs[list(seq_df.columns)] = seq_df
    added = list(seq_df.columns)
 
    beta_col = f"VDJ_1_{SEQ_FIELD}"
    if beta_col in seq_df.columns:
        adata.obs["cdr3_aa"] = seq_df[beta_col]                       # raw junction
        adata.obs["cdr3_aa_trimmed"] = seq_df[beta_col].map(junction_to_cdr3)
        added += ["cdr3_aa", "cdr3_aa_trimmed"]
    return added
 
 
def process_sample(input_dir, subdir, sample, suffix):
    """Full per-sample pipeline: load -> QC -> primary-CDR3 filter -> clonotypes
    -> sample-prefixed ids -> TCR sequence columns. Returns the processed adata."""
    adata = load_sample(input_dir, subdir, sample, suffix)
    sample = str(adata.obs["sample"].iloc[0])
    n_raw = adata.n_obs
 
    ir.pp.index_chains(adata)
    ir.tl.chain_qc(adata)
 
    # keep cells with a primary CDR3 (no chain_pairing filtering)
    if PRIMARY_ARM is not None:
        primary = get_primary_cdr3(adata, PRIMARY_ARM)
    else:
        primary = get_primary_cdr3(adata, "VDJ_1").fillna(get_primary_cdr3(adata, "VJ_1"))
    adata = adata[primary.notna().values].copy()
    adata.obs["primary_cdr3_aa"] = primary.loc[adata.obs_names].values
 
    # clonotype definition WITHIN this sample (patient-private by construction)
    ir.pp.ir_dist(adata, metric="identity", sequence="aa")
 
    # clone_id (auxiliary): define_clonotypes is nt+identity; compute nt dist if needed
    try:
        ir.tl.define_clonotypes(adata, receptor_arms=CLONE_ID_ARMS, dual_ir=DUAL_IR)
    except Exception as e:
        print(f"    [warn] {sample}: define_clonotypes needs nt distances "
              f"({type(e).__name__}); computing ir_dist(nt) then retrying")
        ir.pp.ir_dist(adata, metric="identity", sequence="nt")
        ir.tl.define_clonotypes(adata, receptor_arms=CLONE_ID_ARMS, dual_ir=DUAL_IR)
 
    # cc_aa_identity (FINAL basis): receptor_arms="all" on aa identity
    ir.tl.define_clonotype_clusters(
        adata, sequence="aa", metric="identity",
        receptor_arms=CC_ARMS, dual_ir=DUAL_IR,
    )   # -> adata.obs["cc_aa_identity"]
 
    # sample-prefixed ids (globally unique; keep NaN as NaN)
    cc = adata.obs["cc_aa_identity"]
    clone = adata.obs["clone_id"]
    # The following were same as the original setting in known TCR modality. 
    adata.obs["sample_clone_id"] = (sample + "_" + cc.astype(str)).where(cc.notna())
    adata.obs["sample_clone_id_any"] = (sample + "_" + clone.astype(str)).where(clone.notna())

    # Try with the loose definition of sample_clone_id:
    #adata.obs["sample_clone_id"] = (sample + "_" + clone.astype(str)).where(clone.notna())
    #adata.obs["sample_clone_id_any"] = (sample + "_" + cc.astype(str)).where(cc.notna())
    # per-cell TCR sequence columns -> obs (saved to the output CSV)
    write_tcr_seq_cols(adata)
 
    n_cc = adata.obs["clone_id"].nunique(dropna=True)
    print(f"[{sample}] {n_raw} cells -> {adata.n_obs} with primary CDR3 | {n_cc} clonotypes")
    return adata
 
 
# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def main():
    subdirs = find_subdirs(INPUT_DIR)
    print(f"Found {len(subdirs)} subdirs on disk: {subdirs}")
 
    suffix_map = build_suffix_map(REFERENCE_DF) if REFERENCE_DF else {}
 
    tasks, skipped, used_keys = [], [], set()
    for sub in subdirs:
        key = next((k for k in (sub, sub.removesuffix("_CD45")) if k in suffix_map), None)
        if REFERENCE_DF and key is None:
            skipped.append(sub)
            continue
        sample, suffix = suffix_map.get(key, (None, None))
        used_keys.add(key)
        tasks.append((sub, sample, suffix))
 
    if skipped:
        print(f"  [warn] {len(skipped)} subdir(s) absent from reference df, skipped: {skipped}")
        print(f"         reference samples available: {sorted(suffix_map)}")
    unmatched_samples = sorted(set(suffix_map) - used_keys)
    if REFERENCE_DF and unmatched_samples:
        print(f"  [warn] {len(unmatched_samples)} reference sample(s) had no subdir: "
              f"{unmatched_samples}")
 
    print(f"\nProcessing {len(tasks)} samples")
    adatas = [process_sample(INPUT_DIR, sub, sample, suffix)
              for sub, sample, suffix in tasks]
 
    # Plain concat of already-processed samples (barcodes + ids already unique).
    # ir_dist matrices in .uns/.obsp do NOT survive concat — that is expected;
    # the obs columns (ids, cdr3 sequences) carry all downstream info.
    adata_tcr = ad.concat(adatas, join="outer", index_unique=None, uns_merge=None)
    assert adata_tcr.obs_names.is_unique, "Barcode collision after concat."
    print(f"\nCombined: {adata_tcr.n_obs} cells across {len(tasks)} samples")
 
    # Restore categorical dtypes concat/scirpy may have widened
    for col in ["chain_pairing", "receptor_type", "receptor_subtype", "sample", "study"]:
        if col in adata_tcr.obs:
            adata_tcr.obs[col] = adata_tcr.obs[col].astype("category")
 
    # --- output table ---
    out = adata_tcr.obs.copy()
    out["cell_id"] = adata_tcr.obs_names
    seq_cols = [c for c in out.columns
                if c in ("cdr3_aa", "cdr3_aa_trimmed")
                or c.endswith(("_v_call", f"_{SEQ_FIELD}", "_j_call"))]
    front = (["cell_id", "sample", "study", "sample_clone_id", "cc_aa_identity",
              "clone_id", "sample_clone_id_any", "primary_cdr3_aa",
              "cdr3_aa", "cdr3_aa_trimmed"]
             + [c for c in seq_cols if c not in ("cdr3_aa", "cdr3_aa_trimmed")]
             + ["chain_pairing"])
    front = [c for c in dict.fromkeys(front) if c in out.columns]   # dedup, keep order
    out = out[front + [c for c in out.columns if c not in front]]
 
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, f"{OUT_PREFIX}_clonotypes.csv")
    h5_path = os.path.join(OUTPUT_DIR, f"{OUT_PREFIX}_airr.h5ad")
    out.to_csv(csv_path, index=False)
    # before adata_tcr.write_h5ad(h5_path)
    seq_like = [c for c in adata_tcr.obs.columns
                if c in ("cdr3_aa", "cdr3_aa_trimmed", "primary_cdr3_aa")
                or c.endswith(("_v_call", f"_{SEQ_FIELD}", "_j_call"))]
    for c in seq_like:
        adata_tcr.obs[c] = adata_tcr.obs[c].astype("string").fillna("").astype(str)
    adata_tcr.write_h5ad(h5_path)
 
    n_final = out["sample_clone_id"].nunique()
    n_any = out["sample_clone_id_any"].nunique()
    print(f"\nFinal: {adata_tcr.n_obs} cells | {n_final} clonotypes "
          f"(cc_aa_identity, receptor_arms='all') | {n_any} via clone_id (any)")
    print(f"Wrote {csv_path}")
    print(f"Wrote {h5_path}")
 
 
if __name__ == "__main__":
    main()

/home/lsxgf/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/pixstor/data/lsxgf/miniconda/envs/py-scverse/lib/python3.9/site-packages/anndata/_core/aligned_mapping.py:64: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(


Found 15 subdirs on disk: ['patient10_T01_CD45', 'patient10_T02_CD45', 'patient11_T0_CD45', 'patient12_T0_CD45', 'patient13_T01_CD45', 'patient13_T02_CD45', 'patient1_T0_CD45', 'patient2_T0_CD45', 'patient3_T0_CD45', 'patient4_T0_CD45', 'patient5_T0_CD45', 'patient6_T0_CD45', 'patient7_T0_CD45', 'patient8_T0_CD45', 'patient9_T0_CD45']
Derived suffixes for 15 samples from /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/ICB_chiffelle_5case_inference_6AllasTrain/unified_cell_predictions/raw_cell_scores/scTRP_simclr_rankMin_study_chiffelle.csv

Processing 15 samples
[patient10_T01_CD45] SRR22857631_patient10_T01_CD45_CD8_TRUST4_barcode_airr.tsv: 734 cells -> chiffelle_Patient10_T01


100%|████████████████████████████████████████████████████████████████████████████████████████████| 478/478 [00:00<00:00, 1003.06it/s]


[chiffelle_Patient10_T01] 734 cells -> 558 with primary CDR3 | 456 clonotypes
[patient10_T02_CD45] SRR22857630_patient10_T02_CD45_CD8_TRUST4_barcode_airr.tsv: 651 cells -> chiffelle_Patient10_T02


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 451/451 [00:00<00:00, 983.77it/s]


[chiffelle_Patient10_T02] 651 cells -> 540 with primary CDR3 | 423 clonotypes
[patient11_T0_CD45] SRR22857629_patient11_T0_CD45_CD8_TRUST4_barcode_airr.tsv: 294 cells -> chiffelle_Patient11_T0


... (22 outputs left out)

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 517/517 [00:00<00:00, 989.52it/s]


[chiffelle_Patient8_T0] 992 cells -> 794 with primary CDR3 | 456 clonotypes
[patient9_T0_CD45] SRR22857617_patient9_T0_CD45_CD8_TRUST4_barcode_airr.tsv: 1702 cells -> chiffelle_Patient9_T0


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 828/828 [00:00<00:00, 996.01it/s]
/mnt/pixstor/data/lsxgf/miniconda/envs/py-scverse/lib/python3.9/site-packages/anndata/_core/merge.py:881: ExperimentalFeatureWarning: Outer joins on awkward.Arrays will have different return values in the future.For details, and to offer input, please see:

	https://github.com/scverse/anndata/issues/898
  warn(


[chiffelle_Patient9_T0] 1702 cells -> 1454 with primary CDR3 | 735 clonotypes

Combined: 8484 cells across 15 samples

Final: 8484 cells | 5466 clonotypes (cc_aa_identity, receptor_arms='all') | 4860 via clone_id (any)
Wrote /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_clonotypes.csv
Wrote /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_airr.h5ad


In [2]:
import os 

os.listdir('/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/')

['max', 'raw', 'trust4_chiffelle_clonotypes.csv', 'trust4_chiffelle_airr.h5ad']

In [3]:
import pandas as pd 
clone_file='/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_clonotypes.csv'
clone_df = pd.read_csv(clone_file)

In [4]:
clone_df.head()

,cell_id,sample,study,sample_clone_id,cc_aa_identity,clone_id,sample_clone_id_any,primary_cdr3_aa,cdr3_aa,cdr3_aa_trimmed,...,VJ_2_junction_aa,VJ_2_j_call,VDJ_2_v_call,VDJ_2_junction_aa,VDJ_2_j_call,chain_pairing,receptor_type,receptor_subtype,clone_id_size,cc_aa_identity_size
0,TCACGAAAGCAATCTC-patient10_A_T0_CD45,chiffelle_Patient10_T01,chiffelle,chiffelle_Patient10_T01_0,0,0,chiffelle_Patient10_T01_0,CASSRSGSRYNEQFF,CASSRSGSRYNEQFF,ASSRSGSRYNEQF,...,NaN,NaN,NaN,NaN,NaN,orphan VDJ,TCR,TRA+TRB,2,2
1,TCAGCTCTCTTGTCAT-patient10_A_T0_CD45,chiffelle_Patient10_T01,chiffelle,chiffelle_Patient10_T01_1,1,1,chiffelle_Patient10_T01_1,CAIRLDQGVDGEQYF,CAIRLDQGVDGEQYF,AIRLDQGVDGEQY,...,NaN,NaN,NaN,NaN,NaN,single pair,TCR,TRA+TRB,1,1
2,TTGAACGCACGCATCG-patient10_A_T0_CD45,chiffelle_Patient10_T01,chiffelle,chiffelle_Patient10_T01_2,2,2,chiffelle_Patient10_T01_2,CSAREGYNERFF,CSAREGYNERFF,SAREGYNERF,...,NaN,NaN,NaN,NaN,NaN,single pair,TCR,TRA+TRB,1,1
3,CAAGAAAAGCTGAAAT-patient10_A_T0_CD45,chiffelle_Patient10_T01,chiffelle,chiffelle_Patient10_T01_3,3,3,chiffelle_Patient10_T01_3,CSVLPWEGPDTQYF,CSVLPWEGPDTQYF,SVLPWEGPDTQY,...,NaN,NaN,NaN,NaN,NaN,single pair,TCR,TRA+TRB,2,2
4,AGTGGGACATCCTAGA-patient10_A_T0_CD45,chiffelle_Patient10_T01,chiffelle,chiffelle_Patient10_T01_3,3,3,chiffelle_Patient10_T01_3,CSVLPWEGPDTQYF,CSVLPWEGPDTQYF,SVLPWEGPDTQY,...,NaN,NaN,NaN,NaN,NaN,single pair,TCR,TRA+TRB,2,2


In [5]:
clone_df['sample'].value_counts()

chiffelle_Patient3_T0      2275
chiffelle_Patient9_T0      1454
chiffelle_Patient8_T0       794
chiffelle_Patient2_T0       621
chiffelle_Patient10_T01     558
chiffelle_Patient13_T02     555
chiffelle_Patient10_T02     540
chiffelle_Patient13_T01     442
chiffelle_Patient1_T0       317
chiffelle_Patient4_T0       254
chiffelle_Patient11_T0      230
chiffelle_Patient5_T0       157
chiffelle_Patient12_T0      104
chiffelle_Patient7_T0       100
chiffelle_Patient6_T0        83
Name: sample, dtype: int64

In [6]:
clone_df.shape

(8484, 27)

In [7]:
import scirpy as ir
import scanpy as sc

In [8]:
adata_tcr = sc.read_h5ad('/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_airr.h5ad')

In [9]:
adata_tcr

AnnData object with n_obs × n_vars = 8484 × 0
    obs: 'cell_id', 'sample', 'study', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'primary_cdr3_aa', 'clone_id', 'clone_id_size', 'cc_aa_identity', 'cc_aa_identity_size', 'sample_clone_id', 'sample_clone_id_any', 'VJ_1_v_call', 'VJ_1_junction_aa', 'VJ_1_j_call', 'VDJ_1_v_call', 'VDJ_1_junction_aa', 'VDJ_1_j_call', 'VJ_2_v_call', 'VJ_2_junction_aa', 'VJ_2_j_call', 'VDJ_2_v_call', 'VDJ_2_junction_aa', 'VDJ_2_j_call', 'cdr3_aa', 'cdr3_aa_trimmed'
    obsm: 'airr', 'chain_indices'

In [10]:
n_cc = adata_tcr.obs["cc_aa_identity"].nunique(dropna=True)
n_clone_id = adata_tcr.obs["clone_id"].nunique(dropna=True)

In [11]:
adata_tcr.obs['sample_clone_id'].nunique(dropna=True)

5466

In [12]:
print(n_cc,n_clone_id )

978 771


In [13]:
adata_tcr.obsm['airr']

<Array [[{c_call: None, ...}], ..., [{...}]] type='8484 * var * {c_call: ?s...'>

In [14]:
cdr3_seq_df = ir.get.airr(adata_tcr, ["v_call",'junction_aa','j_call'])

In [15]:
cdr3_seq_df.head()

,VJ_1_v_call,VJ_1_junction_aa,VJ_1_j_call,VDJ_1_v_call,VDJ_1_junction_aa,VDJ_1_j_call,VJ_2_v_call,VJ_2_junction_aa,VJ_2_j_call,VDJ_2_v_call,VDJ_2_junction_aa,VDJ_2_j_call
cell_id,,,,,,,,,,,,
TCACGAAAGCAATCTC-patient10_A_T0_CD45,None,None,None,TRBV12-3*01,CASSRSGSRYNEQFF,TRBJ2-1*01,None,None,None,None,None,None
TCAGCTCTCTTGTCAT-patient10_A_T0_CD45,TRAV8-4*01,CAVSENQFYF,TRAJ49*01,TRBV10-3*02,CAIRLDQGVDGEQYF,TRBJ2-7*01,None,None,None,None,None,None
TTGAACGCACGCATCG-patient10_A_T0_CD45,TRAV9-2*01,CALEGHGGSQGNLIF,TRAJ42*01,TRBV20-1*01,CSAREGYNERFF,TRBJ2-1*01,None,None,None,None,None,None
CAAGAAAAGCTGAAAT-patient10_A_T0_CD45,TRAV26-2*01,CILRYFGNEKLTF,TRAJ48*01,TRBV29-1*01,CSVLPWEGPDTQYF,TRBJ2-3*01,None,None,None,None,None,None
AGTGGGACATCCTAGA-patient10_A_T0_CD45,TRAV26-2*01,CILRYFGNEKLTF,TRAJ48*01,TRBV29-1*01,CSVLPWEGPDTQYF,TRBJ2-3*01,None,None,None,None,None,None


### exactlly match `ENSEMBLE_mode` when predict maxpooling of known TCR data, using Trust4_sample_clone_id instead. 

In [1]:
#!/usr/bin/env python
"""
Standalone end-to-end: Chiffelle Trust4-clonotype branch.

For each of the 6 leave-one-out checkpoints, load the raw (cell-level) Chiffelle
h5ad, take its per-cell score, group cells by Trust4_sample_clone_id (cells with
no TCR match kept as patient-specific singletons), max-pool the score within each
clone, then compute the single ensemble mode `rankMin_study`
(study-wide rank for each fold, min of each rank as score;
per-patient Jenks vote as pred) and write one CSV.

WHAT IS RECONSTRUCTED (not from your shared code — verify against your originals):
  - find_raw_h5ad          : file-discovery layout (glob patterns below)
  - get_sample_col         : which obs column is the patient/sample label
  - _jenks_thresh_simple   : 1-D 2-class natural-break threshold
  - _jenks_vote            : per-column per-group Jenks vote -> majority
  - SCORE_COL_H5AD         : the obs column holding the model score
Everything else is your code verbatim or a direct extract of one mode.

Run:  python chiffelle_trust4_standalone.py \
        --log1p /path/chiffelle_log1p.h5ad \
        --clono trust4_CD45_CD8_clonotypes.csv \
        --score-col score \
        --out chiffelle_trust4_out
"""

import os
import glob
import argparse
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================================
# CONFIG  — EDIT THESE
# ============================================================================
BASE = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein"
RES  = f"{BASE}/results/classifier"
JOBS = f"{BASE}/jobs/classifier/six_studies"   # base path for checkpoints

# scTRP_simclr: one dedicated leave-one-out checkpoint per study (+ TCR clonotype)
CK_SIMCLR = {
    "caushi":   f"{JOBS}/supconloss_simCLR/test_caushi/hvg2000_by_study_6_89_genes/regenerate_pt/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght1.0-SuContinue/model_e3.pt",
    "hanada":   f"{JOBS}/supconloss_simCLR/test_hanada/hvg2000_by_study_6_89_genes/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght0.3-SuContinue/model_e25.pt",
    "lowery":   f"{JOBS}/supconloss_simCLR/test_lowery/hvg2000_by_study_6_89_genes/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght1.0-SuContinue/model_e1.pt",
    "meng":     f"{JOBS}/supconloss_simCLR/test_meng/hvg2000_by_study_6_89_genes_newCK_lessLayer/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght0.3-SuContinue/model_e10.pt",
    "oliveira": f"{JOBS}/supconloss_simCLR/test_oliveira/hvg2000_by_study_6_89_genes_newCK_lessLayer/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght1.0-SuContinue/model_e5.pt",
    "zheng":    f"{JOBS}/supconloss_simCLR/test_zheng/hvg2000_by_study_6_89_genes/regenerate_pt/save/dev_SixStudy_supconH_SimCLR_glenhvg2000_by_study_6_89_genes_unfix1_lr0.0001_lrseq0.0001_npos9_nneg120_t0.1_maxl100_simclr_wght0.3-SuContinue/model_e1.pt",
}
SIX_STUDIES = ["caushi", "hanada", "lowery", "meng", "oliveira", "zheng"]

# Original Chiffelle obs (has sample_clone_id, reactivity, reactivity_fine, sample).
LOG1P_H5AD = {"chiffelle": f"{BASE}/data_collection/chiffelle_melanoma_Immunity/rawCount_adatas_cd8/chiffelle_15samples_CD8_log1p.h5ad",}

TRUST4_CLONO_CSV = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_clonotypes.csv"   # from match/clonotype step

SCORE_COL_H5AD = "pro_jenks_OT_deltarho_30best_score"          # <<< SET to the obs column holding the model score
POOL_TYPES = ["max"]              # you only need max; add "raw"/"mean"/... if wanted
METHOD_PREFIX = "scTRP_simclr"
SAVE_MODE = "rankMin_study"
OUTPUT_ROOT = Path("/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle")

# Trust4 clonotype CSV column names
TRUST4_BARCODE_COL = "cell_id"
TRUST4_ID_COL = "sample_clone_id"

# get_sample_col candidates (first present wins)
SAMPLE_COL_CANDIDATES = ["sample", "patient", "sample_id", "donor", "Sample"]

# find_raw_h5ad search patterns ({dir}=checkpoint dir, {ds}=dataset name).
# Must resolve to the RAW cell-level h5ad (pooling is done explicitly here).
RAW_H5AD_GLOBS = [
    "{dir}/*{ds}*raw*.h5ad",
    "{dir}/*{ds}*cell*.h5ad",
    "{dir}/*{ds}*.h5ad",
    "{dir}/../*{ds}*.h5ad",
]


# ============================================================================
# RECONSTRUCTED HELPERS — verify against your originals
# ============================================================================
def find_raw_h5ad(ck_path, study_prefix):
    ck_dir  = os.path.dirname(ck_path)
    matches = [f for f in glob.glob(os.path.join(ck_dir, f"*{study_prefix}*.h5ad"))
               if "_averagedPRJ" not in f and "_attn" not in f]
    if not matches:
        return None
    matches.sort(key=os.path.getctime)
    return matches[-1]


def get_sample_col(obs):
    """RECONSTRUCTED. Return the patient/sample Series from an obs frame."""
    for c in SAMPLE_COL_CANDIDATES:
        if c in obs.columns:
            return obs[c]
    raise ValueError(
        f"No sample column among {SAMPLE_COL_CANDIDATES}; obs has {list(obs.columns)[:20]}"
    )


def _jenks_thresh_simple(vals):
    """Per-patient Jenks threshold; falls back to median if jenkspy unavailable."""
    try:
        import jenkspy
        return jenkspy.jenks_breaks(list(vals), n_classes=2)[1]
    except Exception:
        return float(np.median(vals))


def _jenks_vote(mat, group_s=None):
    """
    Per-checkpoint Jenks(k=2) binary vote → fraction across checkpoints → majority.

    group_s is None  → one threshold per checkpoint over ALL cells   (study-wide)
    group_s provided → one threshold per checkpoint PER GROUP         (per-patient)

    Returns (vote_fraction, vote_pred). NaN cells vote 0; checkpoints/groups with
    <2 valid values contribute 0 (same convention as the original vote block).
    """
    vm = pd.DataFrame(0.0, index=mat.index, columns=mat.columns)
    for col in mat.columns:
        s = mat[col]
        if group_s is None:
            vals = s.dropna().values
            if len(vals) < 2:
                continue
            thresh = _jenks_thresh_simple(vals)
            vm[col] = (s > thresh).astype(float)          # NaN > thresh → False → 0
        else:
            for pat, grp in s.groupby(group_s):
                vals = grp.dropna().values
                if len(vals) < 2:
                    continue
                thresh = _jenks_thresh_simple(vals)
                vm.loc[grp.index, col] = (grp.values > thresh).astype(float)
    frac = vm.mean(axis=1)
    return frac, (frac > 0.5).astype(int)


# ============================================================================
# YOUR CODE (verbatim)
# ============================================================================
def _pool_score_matrix(score_mat, sample_clone_ids, pool_type):
    """Pool each checkpoint column by clonotype for one pool_type."""
    if pool_type == "raw" or sample_clone_ids is None or sample_clone_ids.isna().all():
        return score_mat.copy()
    pooled = {}
    clones = sample_clone_ids.reindex(score_mat.index)
    for col in score_mat.columns:
        tmp = pd.Series(score_mat[col].values, index=clones.values)
        grp = tmp.groupby(level=0)
        if pool_type == "mean":
            p = grp.transform("mean")
        elif pool_type == "max":
            p = grp.transform("max")
        elif pool_type == "median":
            p = grp.transform("median")
        elif pool_type == "p75":
            p = grp.transform(lambda x: x.quantile(0.75))
        else:
            p = tmp
        pooled[col] = p.values
    return pd.DataFrame(pooled, index=score_mat.index)


# ============================================================================
# ONE MODE (direct extract of zscoreMean_studyPerPatient)
# ============================================================================
def compute_zscore_mean_study_perpatient(score_mat_pooled, sample_s):
    """score = mean over checkpoints of study-wide z-scored pooled scores;
    pred  = per-patient Jenks vote on the same z-scored matrix.
    (NOTE: a checkpoint column that is constant study-wide has std 0 and yields
    NaN z-scores -> those cells drop out downstream; unlikely on real scores.)"""
    zmat = pd.DataFrame(index=score_mat_pooled.index,
                        columns=score_mat_pooled.columns, dtype=float)
    for col in score_mat_pooled.columns:
        s = score_mat_pooled[col]
        zmat[col] = (s - s.mean()) / s.std(ddof=0)
    score = zmat.mean(axis=1)
    _, pred = _jenks_vote(zmat, group_s=sample_s)
    return score, pred


import numpy as np
import pandas as pd
import jenkspy


def compute_rankmin_study(
    score_mat_pooled: pd.DataFrame,
    sample_s: pd.Series,
) -> tuple[pd.Series, pd.Series]:
    """Compute rank-min consensus score and per-sample Jenks prediction.

    Parameters
    ----------
    score_mat_pooled : DataFrame, shape (n_cells, n_models)
        Raw model scores, e.g. columns model1_score … model6_score.
        Ranking is done across ALL cells (pooled), not per sample.
    sample_s : Series, same index as score_mat_pooled
        Sample label for each cell.

    Returns
    -------
    score_s : Series
        rank_min_study — each cell's minimum rank across all model columns.
    pred_s : Series (int, 0/1)
        Per-sample Jenks prediction on rank_min_study.
        pred=1 if rank_min_study > Jenks threshold for that sample, else 0.
    """
    # ── Step 1: rank each model column pooled, then take per-cell minimum ──
    rank_mat = score_mat_pooled.rank(method='average', ascending=True)
    score_s  = rank_mat.min(axis=1).rename('rank_min_study')

    # ── Step 2: per-sample Jenks prediction ────────────────────────────────
    pred_s = pd.Series(0, index=score_s.index, name='pred', dtype=int)

    for samp, idx in sample_s.groupby(sample_s).groups.items():
        scores = score_s.loc[idx].dropna().values.astype(float)

        if len(scores) < 2 or np.unique(scores).size < 2:
            continue  # leave pred=0

        try:
            thresh = jenkspy.jenks_breaks(scores.tolist(), n_classes=2)[1]
            pred_s.loc[idx] = (score_s.loc[idx].values > thresh).astype(int)
        except Exception as e:
            print(f"  [jenks] skip sample '{samp}': {e}")

    return score_s, pred_s


# ============================================================================
# Trust4 clone ids (singleton fakes for unmatched cells)
# ============================================================================
def build_trust4_clone_ids(clono_csv, index, sample_s,
                           id_col=TRUST4_ID_COL, barcode_col=TRUST4_BARCODE_COL):
    """Map Trust4 clonotype ids onto `index` (cell barcodes).

    Unmatched cells are kept (never dropped). Each gets its own singleton id
        f"{sample}__nomatch__{barcode}"
    (patient in front, globally-unique barcode after), so pooling is a no-op for
    it and its raw score is preserved. Returns (trust4_ids, matched_mask)."""
    clono = pd.read_csv(clono_csv, usecols=[barcode_col, id_col])
    bc2clone = (clono.dropna(subset=[barcode_col])
                     .drop_duplicates(subset=[barcode_col])
                     .set_index(barcode_col)[id_col])

    trust4 = bc2clone.reindex(index)
    matched = trust4.notna()

    barcodes = pd.Series(list(index), index=index).astype(str)
    fake = sample_s.reindex(index).astype(str) + "__nomatch__" + barcodes
    trust4 = trust4.where(matched, fake)
    trust4.name = "Trust4_sample_clone_id"

    print(f"    Trust4 clones: {int(matched.sum())} matched cells, "
          f"{int((~matched).sum())} singleton fakes "
          f"({100 * matched.mean():.1f}% matched)")
    return trust4, matched


# ============================================================================
# Extract + save
# ============================================================================
# Sequence columns pulled from the clono CSV for the *_clone_TCRseq.csv merge.
_SEQ_EXACT = ("cdr3_aa", "cdr3_aa_trimmed")
_SEQ_SUFFIXES = ("_v_call", "_j_call", "_junction_aa", "_cdr3_aa")
CLONO_SIZE_COL = "cc_aa_identity_size"       # in CLONO_CSV -> Trust4_clone_size
ANNOT_COLS = ("reactivity", "reactivity_fine", "sample_clone_id")
 
 
def _load_clono_extras(clono_csv, barcode_col=TRUST4_BARCODE_COL):
    """From the clono CSV, return (bc2size, cdr3_seq_df), both keyed by cell_id.
 
    bc2size : Series cell_id -> cc_aa_identity_size  (None if the column is absent)
    cdr3_seq_df : DataFrame with cell_id + all TCR sequence columns present.
    """
    header = pd.read_csv(clono_csv, nrows=0).columns.tolist()
 
    bc2size = None
    if CLONO_SIZE_COL in header:
        sz = pd.read_csv(clono_csv, usecols=[barcode_col, CLONO_SIZE_COL])
        bc2size = (sz.dropna(subset=[barcode_col])
                     .drop_duplicates(subset=[barcode_col])
                     .set_index(barcode_col)[CLONO_SIZE_COL])
    else:
        print(f"    [warn] {CLONO_SIZE_COL!r} absent in clono CSV; "
              f"Trust4_clone_size will be skipped")
 
    seq_cols = [c for c in header if c in _SEQ_EXACT or c.endswith(_SEQ_SUFFIXES)]
    cdr3_seq_df = pd.read_csv(clono_csv, usecols=[barcode_col] + seq_cols)
    return bc2size, cdr3_seq_df
 
 
def extract_and_save_chiffelle_trust4(
        ck_dic, study_list, ori_obs, method_prefix, raw_dir, pool_dirs,
        trust4_clono_csv, score_col, save_mode=SAVE_MODE,
        dataset="chiffelle", annotated=None):
    """Extract ensemble scores, attach Trust4 clonotype ids, write cell / clone /
    clone_TCRseq CSVs for one dataset.
 
    dataset  : name used by find_raw_h5ad and in output filenames (generalises
               beyond 'chiffelle').
    annotated: None -> auto-detect from ori_obs (all of reactivity, reactivity_fine,
               sample_clone_id present). True/False forces the path.
                 True  : label-aware output; clone reactivity = majority vote.
                 False : minimal output (scores/ids only); clone = keep-first cell.
    """
    import scanpy as sc
 
    if annotated is None:
        annotated = all(c in ori_obs.columns for c in ANNOT_COLS)
    present = [c for c in ANNOT_COLS if c in ori_obs.columns]
    print(f"    [{dataset}] mode: {'annotated' if annotated else 'minimal'} "
          f"(label cols present: {present})")
 
    # --- load checkpoint scores ---
    score_frames, ref_obs = [], None
    for study in study_list:
        ck_path = ck_dic.get(study)
        if not ck_path or not os.path.exists(ck_path):
            print(f"    [warn] {method_prefix}/{study}: checkpoint missing")
            continue
        h5ad_path = find_raw_h5ad(ck_path, dataset)
        if h5ad_path is None:
            print(f"    [warn] {method_prefix}/{study}: {dataset} h5ad not found in "
                  f"{os.path.dirname(ck_path)}")
            continue
        adata = sc.read_h5ad(h5ad_path)
        if score_col not in adata.obs.columns:
            print(f"    [warn] {method_prefix}/{study}: {score_col!r} absent")
            continue
        score_frames.append(adata.obs[score_col].rename(study))
        if ref_obs is None:
            ref_obs = adata.obs
 
    if not score_frames or ref_obs is None:
        print(f"    [skip] {method_prefix}/{dataset}: no h5ads loaded")
        return
 
    score_mat = pd.concat(score_frames, axis=1)
    print(f"    {method_prefix}/{dataset}: {len(score_frames)} checkpoints, "
          f"{len(score_mat)} cells")
 
    try:
        sample_s = get_sample_col(ori_obs).reindex(score_mat.index)
    except ValueError:
        sample_s = get_sample_col(ref_obs).reindex(score_mat.index)
 
    trust4_ids, matched = build_trust4_clone_ids(
        trust4_clono_csv, score_mat.index, sample_s)
 
    # clone size + TCR sequences from the clono CSV (both keyed by cell_id)
    bc2size, cdr3_seq_df = _load_clono_extras(trust4_clono_csv)
 
    # label columns (annotated path only)
    if annotated:
        reactivity = (ori_obs["reactivity"].reindex(score_mat.index)
                      .astype(str).str.strip().map({"1": 1, "0": 0, 1: 1, 0: 0}))
        reactivity_fine = ori_obs["reactivity_fine"].reindex(score_mat.index)
        true_clone = ori_obs["sample_clone_id"].reindex(score_mat.index)
 
    for pool_type in POOL_TYPES:
        score_mat_pooled = _pool_score_matrix(score_mat, trust4_ids, pool_type)
        score_s, pred_s = compute_rankmin_study(score_mat_pooled, sample_s)
 
        key = f"{method_prefix}_{save_mode}_trust4"
        out_dir = raw_dir if pool_type == "raw" else pool_dirs[pool_type]
 
        # ── cell-level df ────────────────────────────────────────────────────
        cols = {"score": score_s, "sample_clone_id": trust4_ids,
                "sample": sample_s, "trust4_matched": matched, "pred": pred_s}
        if annotated:
            cols.update(reactivity=reactivity, true_sample_clone_id=true_clone,
                        reactivity_fine=reactivity_fine)
        df = pd.DataFrame(cols, index=score_mat.index)
        df.index.name = "cell_barcode"
 
        # Drops ONLY cells missing score/sample (+reactivity when annotated) —
        # never for lacking a Trust4 match (those carry a fake id + real score).
        drop_on = ["score", "sample"] + (["reactivity"] if annotated else [])
        df = df.dropna(subset=drop_on)
        if annotated:
            df["reactivity"] = df["reactivity"].astype(int)
            df["reactivity_fine"] = df["reactivity_fine"].astype(int)
            order = ["score", "reactivity", "sample_clone_id", "true_sample_clone_id",
                     "sample", "reactivity_fine", "trust4_matched", "pred"]
        else:
            order = ["score", "sample_clone_id", "sample", "trust4_matched", "pred"]
        df = df[order].sort_values("score", ascending=False, kind="stable")
 
        out_cell = out_dir / f"{key}_{dataset}_cell.csv"
        df.to_csv(out_cell)
 
        # ── clone-level df (matched only, one row per Trust4 clone) ───────────
        matched_df = df[df["trust4_matched"]].copy()
        if annotated:
            # majority-vote reactivity (1 wins ties); reactivity_fine = mode among
            # the vote-matching cells, so the two stay consistent.
            matched_df["_react_vote"] = matched_df.groupby("sample_clone_id")["reactivity"].transform(
                lambda s: 1 if s.mean() >= 0.5 else 0)
 
            def _fine_vote(sub):
                pool = sub.loc[sub["reactivity"] == sub["_react_vote"].iloc[0],
                               "reactivity_fine"]
                return int(pool.mode().max())
            fine_map = (matched_df.groupby("sample_clone_id")
                        [["reactivity", "_react_vote", "reactivity_fine"]].apply(_fine_vote))
 
            clone_df = (matched_df.drop_duplicates(subset="sample_clone_id", keep="first")
                        .assign(reactivity=lambda d: d["_react_vote"].astype(int),
                                reactivity_fine=lambda d: d["sample_clone_id"].map(fine_map).astype(int))
                        .drop(columns="_react_vote"))
        else:
            # minimal: keep one representative cell per Trust4 clone
            clone_df = matched_df.drop_duplicates(subset="sample_clone_id", keep="first").copy()
 
        # Trust4_clone_size = scirpy cc_aa_identity_size (per representative cell)
        if bc2size is not None:
            clone_df.insert(clone_df.columns.get_loc("sample_clone_id") + 1,
                            "Trust4_clone_size", clone_df.index.map(bc2size))
 
        out_clone = out_dir / f"{key}_{dataset}_clone.csv"
        clone_df.to_csv(out_clone)
 
        # ── clone + TCR sequences ────────────────────────────────────────────
        t4_clone_seq = (clone_df.reset_index()
                        .merge(cdr3_seq_df, how="left",
                               left_on="cell_barcode", right_on=TRUST4_BARCODE_COL))
        if TRUST4_BARCODE_COL != "cell_barcode" and TRUST4_BARCODE_COL in t4_clone_seq.columns:
            t4_clone_seq = t4_clone_seq.drop(columns=[TRUST4_BARCODE_COL])
        out_clone_seq = out_dir / f"{key}_{dataset}_clone_TCRseq.csv"
        t4_clone_seq.to_csv(out_clone_seq, index=False)
 
        print(f"      [{pool_type}][{save_mode}] cell : {len(df)} rows "
              f"({int(df['trust4_matched'].sum())} matched) -> {out_cell.name}")
        print(f"      [{pool_type}][{save_mode}] clone: {len(clone_df)} clones "
              f"-> {out_clone.name} (+ {out_clone_seq.name})")
  
 



"""
# ============================================================================
# Main
# ============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--log1p", default=LOG1P_H5AD["chiffelle"])
    ap.add_argument("--clono", default=TRUST4_CLONO_CSV)
    ap.add_argument("--score-col", default=SCORE_COL_H5AD)
    ap.add_argument("--out", default=str(OUTPUT_ROOT))
    ap.add_argument("--dataset", default="chiffelle",
                    help="dataset name for find_raw_h5ad + output filenames")
    ap.add_argument("--annotated", default="auto", choices=["auto", "yes", "no"],
                    help="auto-detect labels, or force the annotated / minimal path")
    args = ap.parse_args()
 
    import scanpy as sc
 
    out_root = Path(args.out)
    raw_dir = out_root / "raw"
    pool_dirs = {pt: out_root / pt for pt in POOL_TYPES if pt != "raw"}
    for d in [raw_dir, *pool_dirs.values()]:
        d.mkdir(parents=True, exist_ok=True)
 
    if not os.path.exists(args.log1p):
        raise FileNotFoundError(f"log1p h5ad not found: {args.log1p}")
    ori_obs = sc.read_h5ad(args.log1p, backed="r").obs
 
    annotated = {"auto": None, "yes": True, "no": False}[args.annotated]
    extract_and_save_chiffelle_trust4(
        CK_SIMCLR, SIX_STUDIES, ori_obs, METHOD_PREFIX,
        raw_dir, pool_dirs, args.clono, args.score_col, SAVE_MODE,
        dataset=args.dataset, annotated=annotated,
    )
 


if __name__ == "__main__":
    main()

"""

'\n# ============================================================================\n# Main\n# ============================================================================\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--log1p", default=LOG1P_H5AD["chiffelle"])\n    ap.add_argument("--clono", default=TRUST4_CLONO_CSV)\n    ap.add_argument("--score-col", default=SCORE_COL_H5AD)\n    ap.add_argument("--out", default=str(OUTPUT_ROOT))\n    ap.add_argument("--dataset", default="chiffelle",\n                    help="dataset name for find_raw_h5ad + output filenames")\n    ap.add_argument("--annotated", default="auto", choices=["auto", "yes", "no"],\n                    help="auto-detect labels, or force the annotated / minimal path")\n    args = ap.parse_args()\n \n    import scanpy as sc\n \n    out_root = Path(args.out)\n    raw_dir = out_root / "raw"\n    pool_dirs = {pt: out_root / pt for pt in POOL_TYPES if pt != "raw"}\n    for d in [raw_dir, *pool_dirs.values()

In [2]:
from pathlib import Path
import scanpy as sc

# --- the inputs ---
LOG1P     = LOG1P_H5AD["chiffelle"]
CLONO_CSV = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/trust4_chiffelle_clonotypes.csv"   # from match/clonotype step

SCORE_COL = "pro_jenks_OT_deltarho_30best_score"          # <<< SET to the obs column holding the model score
POOL_TYPES = ["max"]              # you only need max; add "raw"/"mean"/... if wanted

# --- output dirs (what main() sets up) ---
out_root  = Path("/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle")
raw_dir   = out_root / "raw"
pool_dirs = {pt: out_root / pt for pt in POOL_TYPES if pt != "raw"}
for d in [raw_dir, *pool_dirs.values()]:
    d.mkdir(parents=True, exist_ok=True)

# --- reference obs (sample_clone_id, reactivity, reactivity_fine, sample) ---
ori_chif_obs = sc.read_h5ad(LOG1P, backed="r").obs

#annotated = {"auto": None, "yes": True, "no": False}[args.annotated]
# --- run ---
extract_and_save_chiffelle_trust4(
    CK_SIMCLR,          # ck_dic
    SIX_STUDIES,        # study_list
    ori_chif_obs,       # ori_obs
    METHOD_PREFIX,      # "scTRP_simclr"
    raw_dir,            # raw_dir
    pool_dirs,          # pool_dirs
    CLONO_CSV,          # trust4_clono_csv
    SCORE_COL,          # score_col
    SAVE_MODE,          # "rankMin_study"
    dataset='chiffelle',
    annotated='auto'
    
)

    [chiffelle] mode: annotated (label cols present: ['reactivity', 'reactivity_fine', 'sample_clone_id'])
    scTRP_simclr/chiffelle: 6 checkpoints, 16727 cells
    Trust4 clones: 8484 matched cells, 8243 singleton fakes (50.7% matched)
      [max][rankMin_study] cell : 16727 rows (8484 matched) -> scTRP_simclr_rankMin_study_trust4_chiffelle_cell.csv
      [max][rankMin_study] clone: 5466 clones -> scTRP_simclr_rankMin_study_trust4_chiffelle_clone.csv (+ scTRP_simclr_rankMin_study_trust4_chiffelle_clone_TCRseq.csv)


In [3]:
import pandas as pd
from pathlib import Path 
out_root  = Path("/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle")
t4_clone_fi = out_root / 'max'/f'scTRP_simclr_{SAVE_MODE}_trust4_chiffelle_clone_TCRseq.csv'
print(t4_clone_fi)
t4_clone = pd.read_csv(t4_clone_fi)

/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/prediction_plus_Trust4/chiffelle/max/scTRP_simclr_rankMin_study_trust4_chiffelle_clone_TCRseq.csv


In [4]:
t4_clone.shape

(5466, 25)

In [5]:
t4_clone.columns

Index(['cell_barcode', 'score', 'reactivity', 'sample_clone_id',
       'Trust4_clone_size', 'true_sample_clone_id', 'sample',
       'reactivity_fine', 'trust4_matched', 'pred', 'primary_cdr3_aa',
       'cdr3_aa', 'cdr3_aa_trimmed', 'VJ_1_v_call', 'VJ_1_junction_aa',
       'VJ_1_j_call', 'VDJ_1_v_call', 'VDJ_1_junction_aa', 'VDJ_1_j_call',
       'VJ_2_v_call', 'VJ_2_junction_aa', 'VJ_2_j_call', 'VDJ_2_v_call',
       'VDJ_2_junction_aa', 'VDJ_2_j_call'],
      dtype='object')

In [6]:
t4_clone.head(30)

,cell_barcode,score,reactivity,sample_clone_id,Trust4_clone_size,true_sample_clone_id,sample,reactivity_fine,trust4_matched,pred,...,VJ_1_j_call,VDJ_1_v_call,VDJ_1_junction_aa,VDJ_1_j_call,VJ_2_v_call,VJ_2_junction_aa,VJ_2_j_call,VDJ_2_v_call,VDJ_2_junction_aa,VDJ_2_j_call
0,AAACGGGGTCCCGACA-patient3_T0_CD45,15958.5,0,chiffelle_Patient3_T0_3,24,chiffelle_Patient3_T0_18,chiffelle_Patient3_T0,0,True,1,...,NaN,TRBV4-1*01,CASSLTRGEQFF,TRBJ2-1*01,NaN,NaN,NaN,NaN,NaN,NaN
1,AACCATGAGGGAAACA-patient9_T0_CD45,15792.5,0,chiffelle_Patient9_T0_6,110,chiffelle_Patient9_T0_44,chiffelle_Patient9_T0,0,True,1,...,NaN,TRBV6-2*01|TRBV6-3*01,CASRTSYEQFF,TRBJ2-1*01,NaN,NaN,NaN,NaN,NaN,NaN
2,AACCGCGGTCATGCAT-patient8_T0_CD45,15753.5,0,chiffelle_Patient8_T0_343,4,chiffelle_Patient8_T0_47,chiffelle_Patient8_T0,8,True,1,...,NaN,TRBV28*01,CASRRLGQGYNGYTF,TRBJ1-2*01,NaN,NaN,NaN,NaN,NaN,NaN
3,AGGGTGAAGGGCTTGA-patient8_T0_CD45,15749.0,1,chiffelle_Patient8_T0_194,11,chiffelle_Patient8_T0_53,chiffelle_Patient8_T0,1,True,1,...,NaN,TRBV11-2*01,CASSLYVGETQYF,TRBJ2-5*01,NaN,NaN,NaN,NaN,NaN,NaN
4,AACTCTTAGCGATAGC-patient3_T0_CD45,15616.5,1,chiffelle_Patient3_T0_49,38,chiffelle_Patient3_T0_102,chiffelle_Patient3_T0,1,True,1,...,TRAJ13*02,TRBV4-2*01,CASSQEFVQGYEQYF,TRBJ2-7*01,NaN,NaN,NaN,NaN,NaN,NaN
5,AACTCTTCAAGCTGTT-patient3_T0_CD45,15551.0,1,chiffelle_Patient3_T0_32,27,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1,True,1,...,TRAJ47*01,TRBV29-1*01,CSVDRGDGYTF,TRBJ1-2*01,NaN,NaN,NaN,NaN,NaN,NaN
6,AAAGCAATCACATACG-patient3_T0_CD45,15397.5,1,chiffelle_Patient3_T0_15,46,chiffelle_Patient3_T0_37_0,chiffelle_Patient3_T0,1,True,1,...,NaN,TRBV9*01,CASSPLAGLNEQFF,TRBJ2-1*01,NaN,NaN,NaN,NaN,NaN,NaN
7,AGCTTGAGTCTGCGGT-patient8_T0_CD45,15323.0,1,chiffelle_Patient8_T0_210,7,chiffelle_Patient8_T0_16,chiffelle_Patient8_T0,1,True,1,...,NaN,TRBV2*01,CASTTAANTGELFF,TRBJ2-2*01,NaN,NaN,NaN,NaN,NaN,NaN
8,AACTCTTAGGATGCGT-patient3_T0_CD45,15283.5,1,chiffelle_Patient3_T0_44,50,chiffelle_Patient3_T0_61,chiffelle_Patient3_T0,1,True,1,...,TRAJ31*01,TRBV9*01,CASSPLAGLNEQFF,TRBJ2-1*01,NaN,NaN,NaN,NaN,NaN,NaN
9,AAATGCCGTAATCACC-patient3_T0_CD45,15271.5,1,chiffelle_Patient3_T0_16,38,chiffelle_Patient3_T0_54,chiffelle_Patient3_T0,1,True,1,...,NaN,TRBV4-2*01,CASSQEFVQGYEQYF,TRBJ2-7*01,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
os.getcwd()

'/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/scripts/scRNA_TCRBCR_surfaceProtein/classifier/case_study/prediction_plus_TRUST4'

In [8]:
import os 
df1 = pd.read_csv(os.path.join("/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/data_collection/chiffelle_melanoma_Immunity/","mmc3_Su_parsed.csv"))  # Contains TCR_b_chain and TCR_a_chain


# df1 already patient-subset; keep only rows with both chains present
df1 = df1.dropna(subset=["TCR_b_chain", "TCR_a_chain"]).copy()

# junctions keep flanking C...F this time -> trim only the trailing char: parts[1][:-1]
df1["Bcdr3_aa"] = df1["TCR_b_chain"].apply(lambda x: x.split("_")[1][:-1])   # TCR beta
df1["Acdr3_aa"] = df1["TCR_a_chain"].apply(lambda x: x.split("_")[1][:-1])   # TCR alpha

# (alpha, beta) pair -> Tumor-reactivity ; dedup pair so it can't fan out
_pair_react = (df1.dropna(subset=["Acdr3_aa", "Bcdr3_aa"])
                  .drop_duplicates(subset=["Acdr3_aa", "Bcdr3_aa"], keep="first")
                  .set_index(["Acdr3_aa", "Bcdr3_aa"])["Tumor-reactivity"])

# t4_clone primary pair: VJ_1_junction_aa = alpha, VDJ_1_junction_aa = beta
_key = pd.MultiIndex.from_arrays([t4_clone["VJ_1_junction_aa"],
                                  t4_clone["VDJ_1_junction_aa"]])
tumor = _pair_react.reindex(_key)          # Tumor-reactivity per row; NaN = no exact pair match
tumor.index = t4_clone.index

def _to_orig(v):
    if v == "yes":
        return 1
    elif v == "no":
        return 0
    else:                                  # no match (NaN) OR matched-but-not-yes/no
        return 8

t4_clone["original_reactivity"] = tumor.map(_to_orig).astype(int)

# ── report ──
matched = tumor.notna()
other = matched & ~tumor.isin(["yes", "no"])
print(t4_clone["original_reactivity"].value_counts())
print(f"matched pairs: {int(matched.sum())} | "
      f"matched but Tumor-reactivity not yes/no (-> 8): {int(other.sum())}")

8    5359
0      62
1      45
Name: original_reactivity, dtype: int64
matched pairs: 108 | matched but Tumor-reactivity not yes/no (-> 8): 1


In [9]:
t4_clone[['cell_barcode', 'score', 'reactivity', 'sample_clone_id',
       'Trust4_clone_size', 'true_sample_clone_id', 'sample',
       'reactivity_fine', 'trust4_matched', 'pred', 'VJ_1_junction_aa','VDJ_1_junction_aa', 'original_reactivity']].head(30)

,cell_barcode,score,reactivity,sample_clone_id,Trust4_clone_size,true_sample_clone_id,sample,reactivity_fine,trust4_matched,pred,VJ_1_junction_aa,VDJ_1_junction_aa,original_reactivity
0,AAACGGGGTCCCGACA-patient3_T0_CD45,15958.5,0,chiffelle_Patient3_T0_3,24,chiffelle_Patient3_T0_18,chiffelle_Patient3_T0,0,True,1,NaN,CASSLTRGEQFF,8
1,AACCATGAGGGAAACA-patient9_T0_CD45,15792.5,0,chiffelle_Patient9_T0_6,110,chiffelle_Patient9_T0_44,chiffelle_Patient9_T0,0,True,1,NaN,CASRTSYEQFF,8
2,AACCGCGGTCATGCAT-patient8_T0_CD45,15753.5,0,chiffelle_Patient8_T0_343,4,chiffelle_Patient8_T0_47,chiffelle_Patient8_T0,8,True,1,NaN,CASRRLGQGYNGYTF,8
3,AGGGTGAAGGGCTTGA-patient8_T0_CD45,15749.0,1,chiffelle_Patient8_T0_194,11,chiffelle_Patient8_T0_53,chiffelle_Patient8_T0,1,True,1,NaN,CASSLYVGETQYF,8
4,AACTCTTAGCGATAGC-patient3_T0_CD45,15616.5,1,chiffelle_Patient3_T0_49,38,chiffelle_Patient3_T0_102,chiffelle_Patient3_T0,1,True,1,CAAARRGGGYQKVTF,CASSQEFVQGYEQYF,1
5,AACTCTTCAAGCTGTT-patient3_T0_CD45,15551.0,1,chiffelle_Patient3_T0_32,27,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1,True,1,CAVREYGNKLVF,CSVDRGDGYTF,1
6,AAAGCAATCACATACG-patient3_T0_CD45,15397.5,1,chiffelle_Patient3_T0_15,46,chiffelle_Patient3_T0_37_0,chiffelle_Patient3_T0,1,True,1,NaN,CASSPLAGLNEQFF,8
7,AGCTTGAGTCTGCGGT-patient8_T0_CD45,15323.0,1,chiffelle_Patient8_T0_210,7,chiffelle_Patient8_T0_16,chiffelle_Patient8_T0,1,True,1,NaN,CASTTAANTGELFF,8
8,AACTCTTAGGATGCGT-patient3_T0_CD45,15283.5,1,chiffelle_Patient3_T0_44,50,chiffelle_Patient3_T0_61,chiffelle_Patient3_T0,1,True,1,CAENLGVHARLMF,CASSPLAGLNEQFF,1
9,AAATGCCGTAATCACC-patient3_T0_CD45,15271.5,1,chiffelle_Patient3_T0_16,38,chiffelle_Patient3_T0_54,chiffelle_Patient3_T0,1,True,1,NaN,CASSQEFVQGYEQYF,8


In [10]:
tp = ((t4_clone["pred"] == 1) & (t4_clone["original_reactivity"] == 1)).sum()
fn = ((t4_clone["pred"] == 0) & (t4_clone["original_reactivity"] == 1)).sum()
recall_seq = tp / (tp + fn)  
print(recall_seq)

0.8666666666666667


In [11]:
tp

39

In [12]:
tp + fn

45

In [53]:
### conclusion: within the 45 positive clones that sequence has been correctly recovered by trust4, 39 clones were predicted correctly.


In [81]:
(t4_clone["original_reactivity"] == 1).sum()

45

In [82]:
(t4_clone["reactivity"] == 1).sum()

146

In [13]:
tp = ((t4_clone["pred"] == 1) & (t4_clone["reactivity"] == 1)).sum()
fn = ((t4_clone["pred"] == 0) & (t4_clone["reactivity"] == 1)).sum()
recall_t4 = tp / (tp + fn)  
print(recall_t4)

0.7671232876712328


In [14]:
print((tp + fn) , tp)

146 112


In [17]:
### conclusion: within the 146 positive clones that were called by trust4, 112 clones were predicted correctly.


In [15]:
from sklearn.metrics import recall_score
recall = recall_score(t4_clone["reactivity"], t4_clone["pred"])

In [16]:
recall

0.7671232876712328

In [87]:
t4_clone[t4_clone["reactivity"] == 1].shape

(146, 26)

In [18]:
t4_clone_fi_ = out_root / 'max'/f'scTRP_simclr_{SAVE_MODE}_trust4_chiffelle_clone_TCRseq_mapOriginalReactivity.csv'
t4_clone.to_csv(t4_clone_fi_)

In [19]:
# compared with clone level prediction based on the true TCR clonotype:
file_pa = f'/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/case_study/ICB_chiffelle_5case_inference_6AllasTrain/unified_cell_predictions/pooled_scores/max/scTRP_simclr_{SAVE_MODE}_chiffelle.csv'
df_ori = pd.read_csv(file_pa)


In [20]:
df_ori.shape

(16727, 6)

In [21]:
df_ori.head()

,cell_barcode,score,reactivity,sample_clone_id,sample,reactivity_fine
0,AAACGGGGTCAGAGGT-patient12_T0_CD45,1817.0,0,chiffelle_Patient12_T0_7,chiffelle_Patient12_T0,8
1,AACACGTGTACCGAGA-patient12_T0_CD45,1673.0,0,chiffelle_Patient12_T0_27,chiffelle_Patient12_T0,8
2,AACGTTGCACGGCTAC-patient12_T0_CD45,2978.5,0,chiffelle_Patient12_T0_34,chiffelle_Patient12_T0,8
3,AACGTTGGTTGAACTC-patient12_T0_CD45,5457.0,0,chiffelle_Patient12_T0_37,chiffelle_Patient12_T0,8
4,AACGTTGGTTGCGTTA-patient12_T0_CD45,69.0,0,chiffelle_Patient12_T0_38,chiffelle_Patient12_T0,8


In [22]:
df_ori[df_ori['sample_clone_id']=='chiffelle_Patient3_T0_103_1']

,cell_barcode,score,reactivity,sample_clone_id,sample,reactivity_fine
13442,AACTCTTCAAGCTGTT-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1
13629,ACGATGTGTGAACCTT-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1
14123,CAAGATCAGCTAACTC-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1
14455,CCCTCCTGTTCGTGAT-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1
14878,CTACGTCTCAGTACGT-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1
15861,GTTCATTCACACGCTG-patient3_T0_CD45,12094.5,1,chiffelle_Patient3_T0_103_1,chiffelle_Patient3_T0,1


In [23]:
import numpy as np
import pandas as pd
import jenkspy


def add_jenks_pred(df: pd.DataFrame) -> pd.DataFrame:
    """Apply per-sample Jenks natural-breaks prediction on 'score'.

    Groups by 'sample', finds the 2-class Jenks threshold within each group,
    and sets pred=1 for cells above the threshold, pred=0 otherwise.

    Edge cases:
      - Fewer than 2 cells in a sample → pred=0 for all
      - All scores identical (no natural break) → pred=0 for all
      - jenkspy raises any exception → pred=0 for all, warning printed
    """
    df = df.copy()
    df['pred'] = 0  # default

    for samp, grp in df.groupby('sample'):
        scores = grp['score'].dropna().values.astype(float)

        if len(scores) < 2 or np.unique(scores).size < 2:
            continue  # leave pred=0

        try:
            breaks = jenkspy.jenks_breaks(scores.tolist(), n_classes=2)
            thresh = breaks[1]
            df.loc[grp.index, 'pred'] = (scores > thresh).astype(int)
        except Exception as e:
            print(f"  [jenks] skip sample '{samp}': {e}")

    return df


In [24]:
df_ori = add_jenks_pred(df_ori)

In [25]:
df_ori = df_ori.sort_values("score", ascending=False).drop_duplicates(subset="sample_clone_id", keep="first")

In [26]:
df_ori.shape


(9717, 7)

In [27]:
from sklearn.metrics import recall_score
recall_ori = recall_score(df_ori["reactivity"], df_ori["pred"])
print(recall_ori)

0.8571428571428571


In [ ]:
# cell - clone - prediction === recall (pred '1' ) / above table or use the TCR sequence (exact match / cdr3 bc).
# sctrp recall:  no plot. 